In [3]:
import pandas as pd

In [4]:
home_values = pd.read_csv("single_family_home_values.csv")

home_values.shape

(895, 326)

In [6]:
single_family_rents = pd.read_csv("single_family_rents.csv")
all_rental_rents = pd.read_csv("all_rental_rents.csv")

print("Single family rents:", single_family_rents.shape)
print("All rental rents:", all_rental_rents.shape)

Single family rents: (608, 146)
All rental rents: (741, 146)


In [7]:
# filtering each dataset into Utah metro areas 

utah_home_values = home_values[home_values["StateName"] == "UT"].copy()
utah_single_family_rents = single_family_rents[
    single_family_rents["StateName"] == "UT"
].copy()
utah_all_rents = all_rental_rents[
    all_rental_rents["StateName"] == "UT"
].copy()

print("Utah home-value metros:", utah_home_values.shape[0])
print("Utah single family rent metros:", utah_single_family_rents.shape[0])
print("Utah all rental metros:", utah_all_rents.shape[0])

Utah home-value metros: 9
Utah single family rent metros: 8
Utah all rental metros: 9


In [8]:
# what is missing 

print("Home values:")
print(utah_home_values["RegionName"].tolist())

print("\nSingle-family rents:")
print(utah_single_family_rents["RegionName"].tolist())

Home values:
['Salt Lake City, UT', 'Ogden, UT', 'Provo, UT', 'St. George, UT', 'Logan, UT', 'Heber, UT', 'Cedar City, UT', 'Vernal, UT', 'Price, UT']

Single-family rents:
['Salt Lake City, UT', 'Ogden, UT', 'Provo, UT', 'St. George, UT', 'Logan, UT', 'Heber, UT', 'Cedar City, UT', 'Vernal, UT']


In [9]:
shared_metros = sorted(
    set(utah_home_values["RegionName"])
    & set(utah_single_family_rents["RegionName"])
)

print(shared_metros)

['Cedar City, UT', 'Heber, UT', 'Logan, UT', 'Ogden, UT', 'Provo, UT', 'Salt Lake City, UT', 'St. George, UT', 'Vernal, UT']


### Market inclusion note

Price, UT was excluded from the primary comparison because it is not available in Zillow’s single family rent dataset. Since our analysis compares single family rents with single family home values, retaining Price UT would create an incomplete comparison. The remaining eight Utah metropolitan areas have both measures and will be used for the core analysis.

In [10]:
comparison_home_values = utah_home_values[
    utah_home_values["RegionName"].isin(shared_metros)
].copy()

comparison_single_family_rents = utah_single_family_rents[
    utah_single_family_rents["RegionName"].isin(shared_metros)
].copy()

print("Home value comparison markets:", comparison_home_values.shape[0])
print("Single family rent comparison markets:", comparison_single_family_rents.shape[0])

Home value comparison markets: 8
Single family rent comparison markets: 8


In [11]:
id_columns = [
    "RegionID",
    "SizeRank",
    "RegionName",
    "RegionType",
    "StateName"
]

home_date_columns = [
    column for column in comparison_home_values.columns
    if column not in id_columns
]

print("Number of monthly home value columns:", len(home_date_columns))
print("First month:", home_date_columns[0])
print("Last month:", home_date_columns[-1])

Number of monthly home value columns: 321
First month: 2000-01-31 00:00:00
Last month: 2026-09-30 00:00:00


In [13]:
home_long = pd.melt(
    comparison_home_values,
    id_vars=id_columns,
    value_vars=home_date_columns,
    var_name="Date",
    value_name="HomeValue"
)

home_long["Date"] = pd.to_datetime(home_long["Date"])

home_long = home_long.dropna(subset=["HomeValue"]).sort_values(   # dropped missing values -- did NOT set them to 0
    ["RegionName", "Date"]
)

home_long[["RegionName", "Date", "HomeValue"]].head()

,RegionName,Date,HomeValue
870,"Cedar City, UT",2009-01-31,178670.364924
878,"Cedar City, UT",2009-02-28,178175.207517
886,"Cedar City, UT",2009-03-31,177675.618009
894,"Cedar City, UT",2009-04-30,176420.968946
902,"Cedar City, UT",2009-05-31,175084.388314


In [14]:
rent_date_columns = [
    column for column in comparison_single_family_rents.columns
    if column not in id_columns
]

rents_long = pd.melt(
    comparison_single_family_rents,
    id_vars=id_columns,
    value_vars=rent_date_columns,
    var_name="Date",
    value_name="SingleFamilyRent"
)

rents_long["Date"] = pd.to_datetime(rents_long["Date"])

rents_long = rents_long.dropna(subset=["SingleFamilyRent"]).sort_values(
    ["RegionName", "Date"]
)

rents_long[["RegionName", "Date", "SingleFamilyRent"]].head()

,RegionName,Date,SingleFamilyRent
790,"Cedar City, UT",2023-03-31,1499.274962
798,"Cedar City, UT",2023-04-30,1527.205283
806,"Cedar City, UT",2023-05-31,1554.392718
814,"Cedar City, UT",2023-06-30,1551.306961
822,"Cedar City, UT",2023-07-31,1575.459560


In [15]:
rent_coverage = (
    rents_long.groupby("RegionName")["Date"]
    .agg(StartDate="min", EndDate="max", MonthsAvailable="count")
    .sort_values("StartDate")
)

rent_coverage

,StartDate,EndDate,MonthsAvailable
RegionName,,,
"Ogden, UT",2015-01-31,2026-09-30,141
"Provo, UT",2015-01-31,2026-09-30,141
"Salt Lake City, UT",2015-01-31,2026-09-30,141
"St. George, UT",2015-03-31,2026-09-30,139
"Heber, UT",2021-03-31,2026-09-30,67
"Logan, UT",2022-11-30,2026-09-30,47
"Cedar City, UT",2023-03-31,2026-09-30,41
"Vernal, UT",2026-03-31,2026-09-30,5


### Data coverage decision

Vernal, UT was excluded from the core trend and stability analysis because Zillow’s single family rent series contains only five monthly observations for that market. To make comparisons fair across the remaining markets, the primary analysis period will be March 2023 through September 2026. This common period includes seven Utah metropolitan areas with at least 41 monthly rent observations.

In [16]:
analysis_start = pd.Timestamp("2023-03-31")
analysis_end = pd.Timestamp("2026-09-30")

core_metros = rent_coverage[
    rent_coverage["StartDate"] <= analysis_start
].index.tolist()

print("Core comparison metros:")
print(core_metros)
print("\nNumber of core metros:", len(core_metros))

Core comparison metros:
['Ogden, UT', 'Provo, UT', 'Salt Lake City, UT', 'St. George, UT', 'Heber, UT', 'Logan, UT', 'Cedar City, UT']

Number of core metros: 7


In [17]:
home_analysis = home_long[
    (home_long["RegionName"].isin(core_metros))
    & (home_long["Date"] >= analysis_start)
    & (home_long["Date"] <= analysis_end)
].copy()

rent_analysis = rents_long[
    (rents_long["RegionName"].isin(core_metros))
    & (rents_long["Date"] >= analysis_start)
    & (rents_long["Date"] <= analysis_end)
].copy()

print("Home value observations:", home_analysis.shape)
print("Single family rent observations:", rent_analysis.shape)

Home value observations: (301, 7)
Single family rent observations: (299, 7)


In [18]:
# what markets have 42 vs 43 months

rent_analysis.groupby("RegionName")["Date"].agg(
    FirstMonth="min",
    LastMonth="max",
    MonthsAvailable="count"
).sort_values("MonthsAvailable")

,FirstMonth,LastMonth,MonthsAvailable
RegionName,,,
"Cedar City, UT",2023-03-31,2026-09-30,41
"Heber, UT",2023-03-31,2026-09-30,43
"Logan, UT",2023-03-31,2026-09-30,43
"Ogden, UT",2023-03-31,2026-09-30,43
"Provo, UT",2023-03-31,2026-09-30,43
"Salt Lake City, UT",2023-03-31,2026-09-30,43
"St. George, UT",2023-03-31,2026-09-30,43


In [19]:
# the 2 gaps are in Cedar City, lets look deeper in Ceder City real quick 
expected_dates = pd.date_range(
    start=analysis_start,
    end=analysis_end,
    freq="ME"
)

cedar_dates = rent_analysis.loc[
    rent_analysis["RegionName"] == "Cedar City, UT",
    "Date"
]

missing_cedar_dates = expected_dates[
    ~expected_dates.isin(cedar_dates)
]

missing_cedar_dates

DatetimeIndex(['2023-08-31', '2023-11-30'], dtype='datetime64[ns]', freq=None)

### Missing value treatment

Cedar City’s single family rent series was missing values for August 2023 and November 2023. Because each gap is a single month surrounded by observed monthly values, we filled the two values using linear interpolation. An `RentImputed` flag will retain transparency about these estimated observations.

In [20]:
rent_grid = pd.MultiIndex.from_product(
    [core_metros, expected_dates],
    names=["RegionName", "Date"]
).to_frame(index=False)

rent_analysis_complete = rent_grid.merge(
    rent_analysis[["RegionName", "Date", "SingleFamilyRent"]],
    on=["RegionName", "Date"],
    how="left"
)

rent_analysis_complete["RentImputed"] = (
    rent_analysis_complete["SingleFamilyRent"].isna()
)

rent_analysis_complete["SingleFamilyRent"] = (
    rent_analysis_complete.groupby("RegionName")["SingleFamilyRent"]
    .transform(lambda values: values.interpolate())
)

print(rent_analysis_complete.shape)
print("Remaining missing rent values:", rent_analysis_complete["SingleFamilyRent"].isna().sum())
print("Interpolated rent values:", rent_analysis_complete["RentImputed"].sum())

(301, 4)
Remaining missing rent values: 0
Interpolated rent values: 2


In [21]:
# time to merge the data -- then visualize 

analysis_data = home_analysis[
    ["RegionName", "Date", "HomeValue"]
].merge(
    rent_analysis_complete,
    on=["RegionName", "Date"],
    how="inner",
    validate="one_to_one"
)

analysis_data = analysis_data.sort_values(["RegionName", "Date"])

print("Final analysis dataset shape:", analysis_data.shape)

analysis_data.head()

Final analysis dataset shape: (301, 5)


,RegionName,Date,HomeValue,SingleFamilyRent,RentImputed
0,"Cedar City, UT",2023-03-31,394809.458615,1499.274962,False
1,"Cedar City, UT",2023-04-30,394375.657839,1527.205283,False
2,"Cedar City, UT",2023-05-31,394734.580097,1554.392718,False
3,"Cedar City, UT",2023-06-30,394757.906631,1551.306961,False
4,"Cedar City, UT",2023-07-31,394265.271817,1575.459560,False


In [22]:
analysis_data["EstimatedGrossRentYield"] = (
    analysis_data["SingleFamilyRent"] * 12
) / analysis_data["HomeValue"]

analysis_data[
    ["RegionName", "Date", "HomeValue", "SingleFamilyRent", "EstimatedGrossRentYield"]
].head()

,RegionName,Date,HomeValue,SingleFamilyRent,EstimatedGrossRentYield
0,"Cedar City, UT",2023-03-31,394809.458615,1499.274962,0.045570
1,"Cedar City, UT",2023-04-30,394375.657839,1527.205283,0.046470
2,"Cedar City, UT",2023-05-31,394734.580097,1554.392718,0.047254
3,"Cedar City, UT",2023-06-30,394757.906631,1551.306961,0.047157
4,"Cedar City, UT",2023-07-31,394265.271817,1575.459560,0.047951


In [23]:
# latest market snapshot 

latest_date = analysis_data["Date"].max()

latest_snapshot = analysis_data[
    analysis_data["Date"] == latest_date
].copy()

latest_snapshot["EstimatedGrossRentYieldPct"] = (
    latest_snapshot["EstimatedGrossRentYield"] * 100
)

latest_snapshot_display = latest_snapshot[
    [
        "RegionName",
        "HomeValue",
        "SingleFamilyRent",
        "EstimatedGrossRentYieldPct"
    ]
].sort_values("EstimatedGrossRentYieldPct", ascending=False)

latest_snapshot_display.round(
    {
        "HomeValue": 0,
        "SingleFamilyRent": 0,
        "EstimatedGrossRentYieldPct": 2
    }
)

,RegionName,HomeValue,SingleFamilyRent,EstimatedGrossRentYieldPct
171,"Ogden, UT",515123.0,2285.0,5.32
257,"Salt Lake City, UT",577928.0,2507.0,5.21
214,"Provo, UT",556373.0,2279.0,4.92
42,"Cedar City, UT",421345.0,1667.0,4.75
128,"Logan, UT",469843.0,1840.0,4.70
300,"St. George, UT",539906.0,2048.0,4.55
85,"Heber, UT",1210309.0,4228.0,4.19


In [24]:
start_snapshot = analysis_data[
    analysis_data["Date"] == analysis_start
][
    ["RegionName", "HomeValue", "SingleFamilyRent"]
].rename(
    columns={
        "HomeValue": "StartHomeValue",
        "SingleFamilyRent": "StartSingleFamilyRent"
    }
)

growth_summary = latest_snapshot[
    [
        "RegionName",
        "HomeValue",
        "SingleFamilyRent",
        "EstimatedGrossRentYieldPct"
    ]
].merge(
    start_snapshot,
    on="RegionName",
    how="inner"
)

growth_summary["RentGrowthPct"] = (
    (growth_summary["SingleFamilyRent"] / growth_summary["StartSingleFamilyRent"]) - 1
) * 100

growth_summary["HomeValueGrowthPct"] = (
    (growth_summary["HomeValue"] / growth_summary["StartHomeValue"]) - 1
) * 100

growth_summary[
    [
        "RegionName",
        "EstimatedGrossRentYieldPct",
        "RentGrowthPct",
        "HomeValueGrowthPct"
    ]
].sort_values("RentGrowthPct", ascending=False).round(2)

,RegionName,EstimatedGrossRentYieldPct,RentGrowthPct,HomeValueGrowthPct
1,"Heber, UT",4.19,11.57,9.75
0,"Cedar City, UT",4.75,11.19,6.72
5,"Salt Lake City, UT",5.21,11.07,5.80
3,"Ogden, UT",5.32,10.95,7.41
4,"Provo, UT",4.92,10.18,5.08
2,"Logan, UT",4.70,8.28,5.12
6,"St. George, UT",4.55,8.00,1.35


### Preliminary market comparison: yield and growth

This table compares the seven core Utah metropolitan areas during the common analysis period of March 2023 through September 2026.

- **Estimated gross rent yield** compares annualized typical single family rent with typical single family home value. Higher values indicate more rent income relative to estimated home value.
- **Rent growth** shows the percentage change in typical single family rent over the analysis period.
- **Home-value growth** shows the percentage change in typical single family home value over the same period.

The results show that no single market leads every measure. Ogden and Salt Lake City have the highest current estimated gross rent yields, while Heber and Cedar City show the strongest rent growth. These measures will be considered together with rental market stability before identifying markets for further acquisition research.

In [25]:
print("Analysis period:", analysis_data["Date"].min().date(), "to", analysis_data["Date"].max().date())
print("Number of core metros:", analysis_data["RegionName"].nunique())
print("Number of metro month observations:", len(analysis_data))

print("\nMissing values in final analysis dataset:")
print(analysis_data.isna().sum())

Analysis period: 2023-03-31 to 2026-09-30
Number of core metros: 7
Number of metro month observations: 301

Missing values in final analysis dataset:
RegionName                 0
Date                       0
HomeValue                  0
SingleFamilyRent           0
RentImputed                0
EstimatedGrossRentYield    0
dtype: int64


### Clean analytic dataset for the proposed project

The cleaned primary dataset contains monthly observations for seven Utah metropolitan areas from March 2023 through September 2026. Each row represents one metropolitan area in one month and includes typical single family home value, typical single family rent, and estimated gross rent yield.

The dataset contains 301 metro month observations (7 markets × 43 months). Two isolated missing rent values for Cedar City were linearly interpolated and retained in the `RentImputed` flag. Price, UT was excluded because it did not appear in the single family rent dataset, while Vernal, UT was excluded because it had only five monthly rent observations. 